In [1]:
import time

last_llm_time = 0
cached_llm_output = ""

In [2]:
!pip install plotly

In [3]:
!pip install yfinance numpy pandas transformers accelerate

In [4]:
import yfinance as yf
import numpy as np
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
import plotly.graph_objects as go

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ✅ Use small, stable DeepSeek model
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load model safely
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Detect device safely
device = "cuda" if torch.cuda.is_available() else "cpu"

def local_llm(prompt):
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a trading AI. First think, then give final structured answer."
            },
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(text, return_tensors="pt").to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=400,   # 🔥 VERY IMPORTANT
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        return response

    except Exception as e:
        return f"ERROR: {str(e)}"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
def parse_llm_output(text):
    try:
        text = text.strip()

        action = re.search(r"(BUY|SELL|HOLD)", text, re.I)
        confidence = re.search(r"confidence:\s*([0-9.]+)", text)

        conf = float(confidence.group(1)) if confidence else 0.6

        # 🔥 clamp value
        conf = max(0.0, min(conf, 1.0))
        reason = re.search(r"reason:\s*(.+)", text, re.I)

        reason_text = reason.group(1).strip() if reason else ""

        # 🔥 prevent template leakage
        if "explain the decision" in reason_text.lower():
            reason_text = "Uptrend with strong bullish momentum and low risk conditions"
        return {
            "action": action.group(1).upper() if action else "HOLD",
            "confidence": float(confidence.group(1)) if confidence else 0.6,
            "reason": reason.group(1).strip() if reason else "Model reasoning incomplete"
        }
    except:
        return {
            "action": "HOLD",
            "confidence": 0.5,
            "reason": "Parsing failed"
        }

In [8]:
def clean_llm_output(text):

    # Remove everything before assistant
    if "<｜Assistant｜>" in text:
        text = text.split("<｜Assistant｜>")[-1]

    # Remove thinking block if exists
    if "<think>" in text:
        parts = text.split("<think>")
        text = parts[-1]  # keep reasoning

    # Remove incomplete sentences at end
    text = text.strip()

    # Optional: cut overly long text
    text = text[:1000]

    return text

In [9]:
def get_clean_data(symbol):

    df = yf.download(symbol, period="1y", progress=False)

    # 🔥 FIX MULTI-INDEX (VERY IMPORTANT)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.index = pd.to_datetime(df.index)

    df = df[["Close"]].copy()

    df = df.dropna()

    return df

In [10]:
def get_weekly_data(symbol):

    df = yf.download(symbol, period="1y", interval="1wk", auto_adjust=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["ma20"] = df["Close"].rolling(20).mean()
    df["ma50"] = df["Close"].rolling(50).mean()

    df["weekly_trend"] = 0
    df.loc[df["ma20"] > df["ma50"], "weekly_trend"] = 1
    df.loc[df["ma20"] < df["ma50"], "weekly_trend"] = -1

    return df[["weekly_trend"]]

In [11]:
def fix_columns(df):

    if isinstance(df.columns, pd.MultiIndex):

        new_cols = []

        for col in df.columns:
            if col[1] == "" or col[1] is None:
                new_cols.append(col[0])   # indicators
            else:
                new_cols.append(col[0])   # keep only price level

        df.columns = new_cols

    return df

In [12]:
def compute_indicators(df):

    df["returns"] = df["Close"].pct_change()

    df["ma20"] = df["Close"].rolling(20).mean()
    df["ma50"] = df["Close"].rolling(50).mean()



    delta = df["Close"].diff()
    gain = (delta.where(delta > 0, 0)).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

    rs = gain / (loss + 1e-6)
    df["rsi"] = 100 - (100 / (1 + rs))

      # 🔥 ADD RSI signals (NEW)
    df.loc[df["rsi"] < 30, "signal"] = 1
    df.loc[df["rsi"] > 70, "signal"] = -1

    return df

In [13]:
def generate_signals(df):

    df = df.copy()
    df["signal"] = 0

    # =========================
    # 🔥 TREND DETECTION
    # =========================

    trend_up = df["ma20"] > df["ma50"]
    trend_down = df["ma20"] < df["ma50"]

    # =========================
    # 🔥 PULLBACK LOGIC (KEY ADD)
    # =========================

    # Price pulls back near MA20
    pullback_buy = (df["Close"] < df["ma20"]) & (df["Close"] > df["ma50"])
    pullback_sell = (df["Close"] > df["ma20"]) & (df["Close"] < df["ma50"])

    # =========================
    # 🔥 MOMENTUM CONFIRMATION
    # =========================

    # RSI recovering (important)
    rsi_recover_buy = (df["rsi"] > df["rsi"].shift(1)) & (df["rsi"] > 40)
    rsi_recover_sell = (df["rsi"] < df["rsi"].shift(1)) & (df["rsi"] < 60)

    # =========================
    # 🔥 WEEKLY FILTER
    # =========================

    weekly_up = df["weekly_trend"] >= 0
    weekly_down = df["weekly_trend"] <= 0

    # =========================
    # 🟢 BUY (TREND + PULLBACK)
    # =========================

    df.loc[
        trend_up &
        pullback_buy &
        rsi_recover_buy &
        weekly_up,
        "signal"
    ] = 1

    # =========================
    # 🔴 SELL (TREND + PULLBACK)
    # =========================

    df.loc[
        trend_down &
        pullback_sell &
        rsi_recover_sell &
        weekly_down,
        "signal"
    ] = -1

    return df

In [14]:
def get_final_data(symbol):

    df = yf.download(symbol, period="3mo", progress=False)

    df = fix_columns(df)

    df.index = pd.to_datetime(df.index)

    df = df.dropna()

    return df

In [15]:
def research_agent(df):
    return {
        "trend": "UP" if df["ma20"].iloc[-1] > df["ma50"].iloc[-1] else "DOWN",
        "rsi": float(df["rsi"].iloc[-1])
    }

In [16]:
def bull_agent(ctx):
    score = 0
    if ctx["trend"] == "UP":
        score += 1
    if ctx["rsi"] > 60:
        score += 1
    return {"bull_score": score}

In [17]:
def bear_agent(ctx):
    score = 0
    if ctx["trend"] == "DOWN":
        score += 1
    if ctx["rsi"] < 40:
        score += 1
    return {"bear_score": score}

In [18]:
def risk_agent(df):
    vol = df["returns"].std()
    return {"risk": "HIGH" if vol > 0.02 else "LOW"}

In [19]:
def compute_metrics(df):
    returns = df["returns"].dropna()

    sharpe = (returns.mean() / (returns.std() + 1e-6)) * np.sqrt(252)

    return {
        "sharpe": round(float(sharpe), 2),
        "volatility": round(float(returns.std()), 4)
    }

In [20]:
def build_prompt(ctx):
    return f"""
Analyze the market data.

Think step-by-step.

Then provide FINAL structured answer ONLY at the end.

Format:

FINAL ANSWER:
- Trend: ...
- RSI: ...
- Signal: ...
- Risk: ...

FINAL DECISION: BUY or SELL or HOLD

Data:
Trend: {ctx['trend']}
RSI: {ctx['rsi']}
Bull Score: {ctx['bull_score']}
Bear Score: {ctx['bear_score']}
Risk: {ctx['risk']}
Sharpe: {ctx['sharpe']}
"""

In [21]:
import re

def extract_decision(text):
    match = re.search(r"FINAL DECISION:\s*(BUY|SELL|HOLD)", text, re.I)
    return match.group(1).upper() if match else "HOLD"

In [22]:
def extract_final_section(text):

    # Extract only FINAL ANSWER section
    if "FINAL ANSWER:" in text:
        text = text.split("FINAL ANSWER:")[-1]

    return text.strip()

In [23]:
def compute_confidence(ctx, action):

    bull = ctx["bull_score"]
    bear = ctx["bear_score"]
    sharpe = ctx["sharpe"]
    vol = ctx["volatility"]
    risk = ctx["risk"]

    signal_strength = abs(bull - bear)
    signal_score = min(signal_strength / 2, 1.0)

    sharpe_score = max(min(sharpe / 2, 1.0), 0)

    risk_penalty = 0.2 if risk == "HIGH" else 0.0
    vol_penalty = min(vol * 5, 0.3)

    confidence = 0.4 * signal_score + 0.4 * sharpe_score
    confidence -= (risk_penalty + vol_penalty)

    confidence = max(0.1, min(confidence, 0.95))

    if action == "HOLD":
        confidence *= 0.7

    return round(confidence, 2)

In [24]:
def final_llm_agent(ctx):

    prompt = build_prompt(ctx)

    raw = local_llm(prompt)

    print("\n🔍 RAW MODEL OUTPUT:\n", raw)

    clean = extract_final_section(raw)

    action = extract_decision(clean)

    confidence = compute_confidence(ctx, action)

    return {
        "action": action,
        "confidence": confidence,
        "explanation": clean
    }

In [25]:
import plotly.graph_objects as go

def plot_stock_with_signals(df, symbol, ctx=None):

    fig = go.Figure()

    # 📈 Price
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["Close"],
        name="Close Price",
        mode="lines"
    ))

    # 📊 MA lines
    fig.add_trace(go.Scatter(x=df.index, y=df["ma20"], name="MA20"))
    fig.add_trace(go.Scatter(x=df.index, y=df["ma50"], name="MA50"))

    # 🔥 ADD BUY/SELL MARKER (ONLY LAST POINT → SAFE)
    if ctx is not None:

        decision = get_final_decision(ctx)

        last_x = df.index[-1]
        last_y = df["Close"].iloc[-1]

        if decision == "BUY":
            fig.add_trace(go.Scatter(
                x=[last_x],
                y=[last_y],
                mode="markers+text",
                marker=dict(size=12, symbol="triangle-up"),
                text=["BUY"],
                textposition="top center",
                name="BUY Signal"
            ))

        elif decision == "SELL":
            fig.add_trace(go.Scatter(
                x=[last_x],
                y=[last_y],
                mode="markers+text",
                marker=dict(size=12, symbol="triangle-down"),
                text=["SELL"],
                textposition="bottom center",
                name="SELL Signal"
            ))

    fig.update_layout(
        title=f"{symbol} Stock with Signals",
        template="plotly_dark",
        hovermode="x unified"
    )

    fig.show()

In [26]:
def plot_rsi_with_signal(df, symbol, ctx=None):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["rsi"],
        name="RSI",
        mode="lines"
    ))

    fig.add_hline(y=70)
    fig.add_hline(y=30)

    # 🔥 MARKER
    if ctx is not None:


        last_x = df.index[-1]
        last_y = df["rsi"].iloc[-1]

        if decision == "BUY":
            label = "BUY"
        elif decision == "SELL":
            label = "SELL"
        else:
            label = "HOLD"

        fig.add_trace(go.Scatter(
            x=[last_x],
            y=[last_y],
            mode="markers+text",
            marker=dict(size=10),
            text=[label],
            textposition="top center",
            name="Signal"
        ))

    fig.update_layout(
        title=f"{symbol} RSI with Signal",
        template="plotly_dark",
        hovermode="x unified"
    )

    fig.show()

In [27]:
import plotly.graph_objects as go

def plot_backtest(df, symbol):

    fig = go.Figure()

    # 📈 Price
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["Close"],
        name="Close Price",
        mode="lines"
    ))

    # 📊 MA lines
    fig.add_trace(go.Scatter(x=df.index, y=df["ma20"], name="MA20"))
    fig.add_trace(go.Scatter(x=df.index, y=df["ma50"], name="MA50"))

    # 🔥 BUY signals
    buy_signals = df[df["signal"] == 1]

    fig.add_trace(go.Scatter(
        x=buy_signals.index,
        y=buy_signals["Close"],
        mode="markers",
        marker=dict(size=10, symbol="triangle-up"),
        name="BUY"
    ))

    # 🔥 SELL signals
    sell_signals = df[df["signal"] == -1]

    fig.add_trace(go.Scatter(
        x=sell_signals.index,
        y=sell_signals["Close"],
        mode="markers",
        marker=dict(size=10, symbol="triangle-down"),
        name="SELL"
    ))

    fig.update_layout(
        title=f"{symbol} Backtesting Signals",
        template="plotly_dark",
        hovermode="x unified"
    )

    fig.show()

In [28]:
def compute_backtest_performance(df):

    position = 0
    entry_price = 0
    profit = 0

    for i in range(len(df)):

        if df["signal"].iloc[i] == 1 and position == 0:
            position = 1
            entry_price = df["Close"].iloc[i]

        elif df["signal"].iloc[i] == -1 and position == 1:
            profit += df["Close"].iloc[i] - entry_price
            position = 0

    return profit

In [29]:
def detect_regime(df):

    df = df.copy()

    # Trend strength
    df["trend_strength"] = abs(df["ma20"] - df["ma50"])

    # Volatility
    df["volatility"] = df["returns"].rolling(10).std()

    # Regime classification
    df["regime"] = "SIDEWAYS"

    df.loc[df["trend_strength"] > df["Close"] * 0.01, "regime"] = "TREND"

    return df

In [30]:
def position_size(volatility):

    if volatility == 0 or np.isnan(volatility):
        return 5

    # 🔥 capped safe sizing
    size = int(10 / (volatility * 100))

    return max(2, min(size, 10))

In [31]:
def backtest_strategy(df):

    df = df.copy()

    position = 0
    entry_price = 0

    equity = 10000
    equity_curve = []
    trades = []

    max_equity = equity

    # 🔥 pyramiding state
    position_size = 0
    add_count = 0
    max_adds = 2   # max pyramid levels

    for i in range(len(df)):

        price = df["Close"].iloc[i]
        signal = df["signal"].iloc[i]
        vol = df["volatility"].iloc[i] if "volatility" in df.columns else 0.02

        # 🔥 SAFE BASE SIZE
        if vol == 0 or np.isnan(vol):
            base_size = 5
        else:
            base_size = int(8 / (vol * 100))
            base_size = max(2, min(base_size, 8))

        # 🔥 DRAWDOWN CONTROL
        max_equity = max(max_equity, equity)
        drawdown = (equity - max_equity) / max_equity

        if drawdown < -0.2:
            position = 0
            position_size = 0
            add_count = 0
            equity_curve.append(equity)
            continue

        # =====================
        # 🟢 ENTRY
        # =====================
        if signal == 1 and position == 0:
            position = 1
            entry_price = price
            position_size = base_size
            add_count = 0

        # =====================
        # 🔥 PYRAMIDING (ADD TO WINNERS)
        # =====================
        elif position == 1 and add_count < max_adds:

            # add when price moves in our favor (+1.5%)
            if price > entry_price * (1 + 0.015 * (add_count + 1)):

                position_size += base_size * 0.5   # smaller adds
                add_count += 1

        # =====================
        # 🔥 TAKE PROFIT
        # =====================
        elif position == 1 and price > entry_price * 1.025:

            pnl = (price - entry_price) * position_size
            equity += pnl
            trades.append(pnl)

            position = 0
            position_size = 0
            add_count = 0

        # =====================
        # 🔥 STOP LOSS
        # =====================
        elif position == 1 and price < entry_price * 0.985:

            pnl = (price - entry_price) * position_size
            equity += pnl
            trades.append(pnl)

            position = 0
            position_size = 0
            add_count = 0

        # =====================
        # 🔴 EXIT SIGNAL
        # =====================
        elif signal == -1 and position == 1:

            pnl = (price - entry_price) * position_size
            equity += pnl
            trades.append(pnl)

            position = 0
            position_size = 0
            add_count = 0

        equity_curve.append(equity)

    df["equity"] = equity_curve

    return df, trades

In [32]:
def add_weekly_trend(df, symbol):

    weekly = get_weekly_data(symbol)

    # align weekly trend to daily index
    weekly = weekly.reindex(df.index, method="ffill")

    df = df.copy()
    df["weekly_trend"] = weekly["weekly_trend"]

    return df

In [33]:
import numpy as np

def compute_performance(trades, df):

    total_profit = sum(trades)

    wins = [t for t in trades if t > 0]
    losses = [t for t in trades if t <= 0]

    win_rate = len(wins) / len(trades) if trades else 0

    returns = df["equity"].pct_change().dropna()

    sharpe = returns.mean() / returns.std() if returns.std() != 0 else 0

    # Max Drawdown
    cum_max = df["equity"].cummax()
    drawdown = (df["equity"] - cum_max) / cum_max
    max_dd = drawdown.min()

    return {
        "Total Profit": round(total_profit, 2),
        "Win Rate": round(win_rate, 2),
        "Sharpe": round(sharpe, 2),
        "Max Drawdown": round(max_dd, 2),
        "Trades": len(trades)
    }

In [34]:
import plotly.graph_objects as go

def plot_equity_curve(df, symbol):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index,
        y=df["equity"],
        mode="lines",
        name="Equity Curve"
    ))

    fig.update_layout(
        title=f"{symbol} Portfolio Growth",
        template="plotly_dark",
        hovermode="x unified"
    )

    fig.show()

In [35]:
def get_ohlc_data(symbol):
    import yfinance as yf
    import pandas as pd

    df = yf.download(symbol, period="1y")

    # 🔥 Flatten MultiIndex safely
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # 🔥 Ensure all required columns exist
    required_cols = ["Open", "High", "Low", "Close", "Volume"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{col} missing in OHLC data")

    df = df.dropna()

    return df

In [36]:
def add_bollinger_bands(df):

    df = df.copy()

    df["bb_mid"] = df["Close"].rolling(window=20, min_periods=1).mean()
    df["bb_std"] = df["Close"].rolling(window=20, min_periods=1).std()

    df["bb_upper"] = df["bb_mid"] + 2 * df["bb_std"]
    df["bb_lower"] = df["bb_mid"] - 2 * df["bb_std"]

    return df

In [37]:
from plotly.graph_objects import Figure, Candlestick, Scatter
import numpy as np

def detect_support_resistance(df, window=10):
    supports = []
    resistances = []

    for i in range(window, len(df)-window):
        low_range = df["Low"][i-window:i+window]
        high_range = df["High"][i-window:i+window]

        if df["Low"][i] == low_range.min():
            supports.append((df.index[i], df["Low"][i]))

        if df["High"][i] == high_range.max():
            resistances.append((df.index[i], df["High"][i]))

    return supports, resistances


import numpy as np

def plot_candlestick(df, symbol):
    fig = go.Figure()

    if not all(col in df.columns for col in ["Open","High","Low","Close"]):
        return go.Figure().update_layout(title="Missing OHLC data")

    # =========================
    # 🕯️ Candlestick
    # =========================
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df["Open"],
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        name="Candles"
    ))

    # =========================
    # 📈 Moving Averages
    # =========================
    if "ma20" in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df["ma20"], name="MA20"))

    if "ma50" in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df["ma50"], name="MA50"))

    # =========================
    # 📉 Trendline
    # =========================
    x = np.arange(len(df))
    y = df["Close"].values
    slope, intercept = np.polyfit(x, y, 1)
    trend = slope * x + intercept

    fig.add_trace(go.Scatter(
        x=df.index,
        y=trend,
        mode="lines",
        name="Trendline"
    ))

    # =========================
    # 🔥 Fibonacci Retracement
    # =========================
    high = df["High"].max()
    low = df["Low"].min()

    levels = [0.236, 0.382, 0.5, 0.618, 0.786]

    for level in levels:
        price = high - (high - low) * level
        fig.add_shape(
            type="line",
            x0=df.index[0],
            x1=df.index[-1],
            y0=price,
            y1=price,
            line=dict(dash="dot"),
        )

    # =========================
    # 🔄 Multi-Timeframe Overlay
    # =========================
    df_htf = df.resample("5D").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last"
    }).dropna()

    fig.add_trace(go.Scatter(
        x=df_htf.index,
        y=df_htf["Close"],
        mode="lines",
        name="HTF Trend"
    ))

    # =========================
    # 🎯 Layout (TradingView Style)
    # =========================
    fig.update_layout(
        template="plotly_dark",
        hovermode="x unified",
        xaxis_rangeslider_visible=False,
        dragmode="drawline",
        newshape=dict(line_color="yellow")
    )

    return fig

In [38]:
def show_dashboard(metrics):

    print("\n📊 TRADING PERFORMANCE DASHBOARD\n")

    for k, v in metrics.items():
        print(f"{k}: {v}")

In [39]:
def run_system(symbol):

    df = get_clean_data(symbol)

    if df is None or len(df) < 60:
        return None, {"error": "Not enough data"}

    df = compute_indicators(df)

    research = research_agent(df)
    bull = bull_agent(research)
    bear = bear_agent(research)
    risk = risk_agent(df)
    metrics = compute_metrics(df)

    ctx = {**research, **bull, **bear, **risk, **metrics}

    decision = final_llm_agent(ctx)

    return ctx, decision

In [40]:
symbol = input("Enter stock (e.g. AAPL, TSLA, INFY.NS): ").upper()



df = get_clean_data(symbol)
df = compute_indicators(df)

# 👉 DO NOT DROP BEFORE PLOTTING
plot_stock_with_signals(df, symbol)

plot_rsi_with_signal(df, symbol)

df = detect_regime(df)
df = add_weekly_trend(df, symbol)

# 🔥 ADD SIGNALS
df = generate_signals(df)



# 📊 backtest
df_bt, trades = backtest_strategy(df)

# 📈 plot signals
plot_backtest(df_bt, symbol)

# 📈 equity curve
plot_equity_curve(df_bt, symbol)

# 📊 performance metrics

# 💰 Profit
#profit = compute_backtest_performance(df)
#print("\n💰 Backtest Profit:", round(profit, 2))

# 👉 Clean only for AI
df_clean = df.dropna(subset=["rsi"])

ctx, decision = run_system(symbol)



if ctx:
    print(f"\n📊 STOCK: {symbol}")

    print("\n📊 METRICS:")
    for k, v in ctx.items():
        print(f"{k}: {v}")

    print("\n🤖 LLM ANALYSIS:")
    print(decision["explanation"])

    print("\n📈 FINAL DECISION:")
    print(f"Action: {decision['action']}")
    print(f"Confidence: {decision['confidence']}")


metrics = compute_performance(trades, df_bt)

print("\nSignal distribution:")
print(df["signal"].value_counts())

show_dashboard(metrics)


Enter stock (e.g. AAPL, TSLA, INFY.NS): TSLA


/tmp/ipykernel_813/1048313898.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="1y", progress=False)


[*********************100%***********************]  1 of 1 completed


/tmp/ipykernel_813/1048313898.py:3: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🔍 RAW MODEL OUTPUT:
 You are a trading AI. First think, then give final structured answer.<｜User｜>
Analyze the market data.

Think step-by-step.

Then provide FINAL structured answer ONLY at the end.

Format:

FINAL ANSWER:
- Trend: ...
- RSI: ...
- Signal: ...
- Risk: ...

FINAL DECISION: BUY or SELL or HOLD

Data:
Trend: UP
RSI: 43.60380582411357
Bull Score: 1
Bear Score: 0
Risk: HIGH
Sharpe: 0.73
<｜Assistant｜><think>
Okay, so I'm trying to figure out how to analyze this market data using the trading AI I'm working with. Let me start by breaking down the information I have.

First, the trend is UP. That means the market is moving in the positive direction, which is generally considered a good sign for potential gains. But I shouldn't jump to conclusions yet; I need to look at other factors to make a well-informed decision.

Next, the RSI is 43.60380582411357. RSI stands for Relative Strength Index, and it's a momentum indicator that measures the relationship between the price of an 

In [41]:
import gradio as gr
import plotly.graph_objects as go
import io, sys, time

# =========================
# 🔥 GLOBAL LIVE CACHE
# =========================
last_llm_time = 0
cached_llm_output = ""

# =========================
# 📊 GRAPH FUNCTIONS
# =========================

def plot_price(df, symbol):
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=df.index, y=df["Close"], name="Price"))
    fig.add_trace(go.Scatter(x=df.index, y=df["ma20"], name="MA20"))
    fig.add_trace(go.Scatter(x=df.index, y=df["ma50"], name="MA50"))

    buys = df[df["signal"] == 1]
    fig.add_trace(go.Scatter(
        x=buys.index, y=buys["Close"],
        mode="markers", marker=dict(size=10, symbol="triangle-up"),
        name="BUY"
    ))

    sells = df[df["signal"] == -1]
    fig.add_trace(go.Scatter(
        x=sells.index, y=sells["Close"],
        mode="markers", marker=dict(size=10, symbol="triangle-down"),
        name="SELL"
    ))

    fig.update_layout(template="plotly_dark", hovermode="x unified")
    return fig


def plot_candlestick(df, symbol):
    fig = go.Figure()

    if not all(col in df.columns for col in ["Open","High","Low","Close"]):
        return go.Figure().update_layout(title="Missing OHLC data")

    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df["Open"],
        high=df["High"],
        low=df["Low"],
        close=df["Close"],
        name="Candles"
    ))

    if "ma20" in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df["ma20"], name="MA20"))

    if "ma50" in df.columns:
        fig.add_trace(go.Scatter(x=df.index, y=df["ma50"], name="MA50"))

    # 🔥 Volume (fixed)
    if "Volume" in df.columns:
        fig.add_trace(go.Bar(
            x=df.index,
            y=df["Volume"],
            name="Volume",
            opacity=0.3
        ))

    fig.update_layout(
        template="plotly_dark",
        hovermode="x unified",
        xaxis_rangeslider_visible=False
    )

    return fig


def plot_rsi_chart(df):
    fig = go.Figure()

    if "rsi" not in df.columns:
        return go.Figure().update_layout(title="RSI not available")

    fig.add_trace(go.Scatter(x=df.index, y=df["rsi"], name="RSI"))
    fig.add_hline(y=70)
    fig.add_hline(y=30)

    fig.update_layout(template="plotly_dark")
    return fig


def plot_equity(df):
    fig = go.Figure()

    if "equity" not in df.columns:
        return go.Figure().update_layout(title="Equity not available")

    fig.add_trace(go.Scatter(x=df.index, y=df["equity"], name="Equity"))
    fig.update_layout(template="plotly_dark")

    return fig

from contextlib import redirect_stdout
import io

def safe_run_llm(symbol):
    buffer = io.StringIO()

    with redirect_stdout(buffer):
        ctx, decision = run_system(symbol)

    output = buffer.getvalue()

    return ctx, decision, output
# =========================
# 🚀 MAIN PIPELINE (LIVE SAFE)
# =========================

def run_full_system(symbol):

    global last_llm_time, cached_llm_output

    try:
        # =========================
        # 🔹 ORIGINAL PIPELINE
        # =========================
        df = get_clean_data(symbol)
        df = compute_indicators(df)
        df = detect_regime(df)
        df = add_weekly_trend(df, symbol)
        df = generate_signals(df)

        df_bt, trades = backtest_strategy(df)

        # =========================
        # 🔥 SAFE OHLC PIPELINE
        # =========================
        # =========================

        # 🔥 SAFE OHLC PIPELINE (FINAL FIX)
        # =========================
        df_ohlc = get_ohlc_data(symbol)

        # =========================
        # 🔥 STANDARDIZE COLUMN NAMES
        # =========================
        df_ohlc.columns = [col.capitalize() for col in df_ohlc.columns]

        # =========================
        # 🔥 KEEP ONLY REQUIRED (NO VOLUME, NO CLOSE DUPLICATE)
        # =========================
        required_cols = ["Open", "High", "Low"]

        missing = [col for col in required_cols if col not in df_ohlc.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        df_ohlc = df_ohlc[required_cols]

        # =========================
        # 🔥 SAFE MERGE (NO OVERLAP NOW)
        # =========================
        df_merged = df_bt.join(df_ohlc, how="inner")

        # =========================
        # 🔥 FINAL CLEAN
        # =========================
        df_merged = df_merged.dropna(subset=["Open", "High", "Low", "Close"])

        # =========================
        # 🔥 FINAL DATA
        # =========================
        df_ohlc_final = df_merged.copy()

        # =========================
        # 📊 METRICS
        # =========================
        metrics = compute_performance(trades, df_bt)
        metrics_text = "\n".join([f"{k}: {v}" for k,v in metrics.items()])

        # =========================
        # 🤖 LLM (RUN EVERY 60s)
        # =========================
        current_time = time.time()

        if current_time - last_llm_time > 60:

            old_stdout = sys.stdout
            sys.stdout = buffer = io.StringIO()

            ctx, decision = run_system(symbol)

            if ctx:
                print(f"\n📊 STOCK: {symbol}")
                print("\n📊 METRICS:")
                for k, v in ctx.items():
                    print(f"{k}: {v}")

                print("\n🤖 LLM ANALYSIS:")
                print(decision["explanation"])

                print("\n📈 FINAL DECISION:")
                print(f"Action: {decision['action']}")
                print(f"Confidence: {decision['confidence']}")

            sys.stdout = old_stdout

            cached_llm_output = buffer.getvalue()
            last_llm_time = current_time

        # =========================
        # 📈 GRAPHS (20s refresh)
        # =========================
        price_fig = plot_price(df_bt, symbol)
        candle_fig = plot_candlestick(df_ohlc_final, symbol)
        rsi_fig = plot_rsi_chart(df_bt)
        equity_fig = plot_equity(df_bt)

        return price_fig, candle_fig, rsi_fig, equity_fig, metrics_text, cached_llm_output

    except Exception as e:
        return None, None, None, None, f"Error: {str(e)}", ""


# =========================
# 🔁 LIVE REFRESH FUNCTION
# =========================

def refresh_system(symbol, live):
    if not live:
        return gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()

    return run_full_system(symbol)



# =========================
# 🎨 FUTURISTIC UI
# =========================

custom_css = """
/* =========================
   🌌 GLOBAL BACKGROUND
========================= */
body {
    background: radial-gradient(circle at top, #0a0a0f, #020204 70%);
    color: #e9d5ff;
    font-family: 'Segoe UI', sans-serif;
}

/* =========================
   🧠 TITLE (NEON PURPLE)
========================= */
h1 {
    text-align: center;
    font-size: 42px;
    font-weight: 700;
    background: linear-gradient(90deg, #c084fc, #7c3aed);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

/* =========================
   📦 MAIN SECTIONS
========================= */
.section {
    background: linear-gradient(145deg, rgba(124,58,237,0.12), rgba(30,27,75,0.25));
    border-radius: 16px;
    padding: 18px;
    margin-bottom: 18px;
    backdrop-filter: blur(14px);
    box-shadow: 0 0 30px rgba(124,58,237,0.15);
    transition: all 0.3s ease-in-out;
    border: 1px solid rgba(124,58,237,0.2);
}

.section:hover {
    transform: translateY(-3px);
    box-shadow: 0 0 40px rgba(124,58,237,0.35);
}

/* =========================
   📊 SUB PANELS
========================= */
.subsection {
    background: rgba(124,58,237,0.08);
    border-radius: 12px;
    padding: 12px;
    border: 1px solid rgba(124,58,237,0.15);
}

/* =========================
   🔘 BUTTON STYLE
========================= */
button {
    border-radius: 12px !important;
    font-weight: 600 !important;
    background: linear-gradient(135deg, #7c3aed, #c084fc) !important;
    color: white !important;
    border: none !important;
    box-shadow: 0 0 15px rgba(124,58,237,0.6);
    transition: all 0.25s ease;
}

button:hover {
    transform: scale(1.05);
    box-shadow: 0 0 25px rgba(192,132,252,0.9);
}

/* =========================
   📥 INPUT BOXES
========================= */
textarea, input {
    background: rgba(20, 20, 35, 0.8) !important;
    color: #e9d5ff !important;
    border: 1px solid rgba(124,58,237,0.3) !important;
    border-radius: 10px !important;
}

/* =========================
   📊 PLOT AREA GLOW
========================= */
.plot-container {
    border-radius: 12px;
    box-shadow: 0 0 25px rgba(124,58,237,0.2);
}

/* =========================
   🧾 TEXTBOX OUTPUT (LLM/METRICS)
========================= */
textarea {
    font-family: monospace;
    font-size: 14px;
}

/* =========================
   🌈 SCROLLBAR (OPTIONAL)
========================= */
::-webkit-scrollbar {
    width: 8px;
}
::-webkit-scrollbar-thumb {
    background: #7c3aed;
    border-radius: 10px;
}
"""
with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as app:

    # =========================
    # HEADER
    # =========================


    gr.Markdown("# 🚀 AI Trading Dashboard")

    # =========================
    # CONTROL BAR
    # =========================
    with gr.Row():
        symbol_input = gr.Textbox(
            label="Stock Symbol",
            placeholder="Enter symbol (e.g. AAPL, TSLA)",
            scale=3
        )
        run_btn = gr.Button("▶ Run", scale=1)
        live_toggle = gr.Checkbox("🔴 Live Mode", value=False, scale=1)

    # =========================
    # 🟦 SECTION 1 — CANDLESTICK (PRIMARY)
    # =========================
    with gr.Group(elem_classes="section"):
        gr.Markdown("## 🟦 Market View")
        candle_chart = gr.Plot()

    # =========================
    # 🟩 SECTION 2 — BACKTESTING
    # =========================
    with gr.Group(elem_classes="section"):
        gr.Markdown("## 🟩 Strategy Performance")

        # Price chart (main backtest view)
        with gr.Group(elem_classes="subsection"):
            gr.Markdown("### 📈 Price & Signals")
            price_chart = gr.Plot()

        # RSI + Equity side-by-side
        with gr.Row():
            with gr.Group(elem_classes="subsection"):
                gr.Markdown("### 📊 RSI")
                rsi_chart = gr.Plot()

            with gr.Group(elem_classes="subsection"):
                gr.Markdown("### 💰 Equity Curve")
                equity_chart = gr.Plot()

    # =========================
    # 🟨 SECTION 3 — INTELLIGENCE
    # =========================
    with gr.Group(elem_classes="section"):
        gr.Markdown("## 🟨 AI Insights")

        with gr.Row():
            with gr.Column():
                gr.Markdown("### 📊 Metrics")
                metrics_box = gr.Textbox(
                    lines=12,
                    show_label=False
                )

            with gr.Column():
                gr.Markdown("### 🤖 LLM Analysis")
                llm_box = gr.Textbox(
                    lines=12,
                    show_label=False
                )

    # =========================
    # BUTTON ACTION
    # =========================
    run_btn.click(
        fn=run_full_system,
        inputs=symbol_input,
        outputs=[
            price_chart,
            candle_chart,
            rsi_chart,
            equity_chart,
            metrics_box,
            llm_box
        ]
    )

    # =========================
    # LIVE MODE (UNCHANGED)
    # =========================
    timer = gr.Timer(20.0)

    timer.tick(
        fn=refresh_system,
        inputs=[symbol_input, live_toggle],
        outputs=[
            price_chart,
            candle_chart,
            rsi_chart,
            equity_chart,
            metrics_box,
            llm_box
        ]
    )

    # New Gradio versions
app.launch()


/tmp/ipykernel_813/1169327142.py:348: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_813/1169327142.py:348: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6d456f3e7731e63df9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [42]:
import gradio as gr
import plotly.graph_objects as go
import io, sys

# =========================
# 📊 GRAPH FUNCTIONS (IMPROVED)
# =========================

def plot_price(df, symbol):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index, y=df["Close"],
        name="Price", mode="lines"
    ))

    fig.add_trace(go.Scatter(
        x=df.index, y=df["ma20"],
        name="MA20", line=dict(dash="dot")
    ))

    fig.add_trace(go.Scatter(
        x=df.index, y=df["ma50"],
        name="MA50", line=dict(dash="dot")
    ))

    # Buy signals
    buys = df[df["signal"] == 1]
    fig.add_trace(go.Scatter(
        x=buys.index, y=buys["Close"],
        mode="markers",
        marker=dict(size=10, symbol="triangle-up"),
        name="BUY"
    ))

    # Sell signals
    sells = df[df["signal"] == -1]
    fig.add_trace(go.Scatter(
        x=sells.index, y=sells["Close"],
        mode="markers",
        marker=dict(size=10, symbol="triangle-down"),
        name="SELL"
    ))

    fig.update_layout(
        template="plotly_dark",
        title=f"{symbol} Price Chart",
        hovermode="x unified",   # 🔥 LIVE HOVER STATS
        height=500
    )

    return fig


def plot_rsi(df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index, y=df["rsi"], name="RSI"
    ))

    fig.add_hline(y=70)
    fig.add_hline(y=30)

    fig.update_layout(
        template="plotly_dark",
        title="RSI Indicator",
        hovermode="x unified"
    )

    return fig


def plot_equity(df):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df.index, y=df["equity"], name="Equity"
    ))

    fig.update_layout(
        template="plotly_dark",
        title="Equity Curve",
        hovermode="x unified"
    )

    return fig


# =========================
# 🚀 MAIN PIPELINE
# =========================

def run_full_system(symbol):

    try:
        df = get_clean_data(symbol)
        df = compute_indicators(df)
        df = detect_regime(df)
        df = add_weekly_trend(df, symbol)
        df = generate_signals(df)

        df_bt, trades = backtest_strategy(df)

        # Metrics
        metrics = compute_performance(trades, df_bt)
        metrics_text = "\n".join([f"{k}: {v}" for k,v in metrics.items()])

        # =========================
        # 🤖 CAPTURE LLM OUTPUT
        # =========================
        old_stdout = sys.stdout
        sys.stdout = buffer = io.StringIO()

        ctx, decision = run_system(symbol)

        if ctx:
            print(f"\n📊 STOCK: {symbol}")
            print("\n📊 METRICS:")
            for k, v in ctx.items():
                print(f"{k}: {v}")

            print("\n🤖 LLM ANALYSIS:")
            print(decision["explanation"])

            print("\n📈 FINAL DECISION:")
            print(f"Action: {decision['action']}")
            print(f"Confidence: {decision['confidence']}")

        sys.stdout = old_stdout
        llm_output = buffer.getvalue()

        # Graphs
        price_fig = plot_price(df_bt, symbol)
        rsi_fig = plot_rsi(df_bt)
        equity_fig = plot_equity(df_bt)

        return price_fig, rsi_fig, equity_fig, metrics_text, llm_output

    except Exception as e:
        return None, None, None, f"Error: {str(e)}", ""


# =========================
# 🎨 PROFESSIONAL UI DESIGN
# =========================

with gr.Blocks(
    theme=gr.themes.Soft(),
    css="""
    body {background: #0f172a;}
    .gradio-container {background: #0f172a;}
    h1 {text-align:center; color:#00f5ff;}
    .box {border-radius:15px; padding:10px;}
    """
) as app:

    gr.Markdown("# 🚀 AI Trading Dashboard")
    gr.Markdown("### Professional Multi-Agent Trading System")

    # 🔹 INPUT
    with gr.Row():
        symbol_input = gr.Textbox(
            label="Stock Symbol",
            placeholder="AAPL / TSLA / INFY.NS"
        )
        run_btn = gr.Button("Run Analysis")

    # =========================
    # 📊 TABS SYSTEM
    # =========================

    with gr.Tabs():

        # 📊 CHARTS TAB
        with gr.Tab("📊 Charts"):
            price_chart = gr.Plot()
            with gr.Row():
                rsi_chart = gr.Plot()
                equity_chart = gr.Plot()

        # 📈 METRICS TAB
        with gr.Tab("📈 Metrics"):
            metrics_box = gr.Textbox(
                lines=15,
                label="Performance Metrics"
            )

        # 🤖 LLM TAB
        with gr.Tab("🤖 LLM Analysis"):
            llm_box = gr.Textbox(
                lines=20,
                label="AI Decision Output"
            )

    # =========================
    # 🔘 BUTTON ACTION
    # =========================

    run_btn.click(
        fn=run_full_system,
        inputs=symbol_input,
        outputs=[
            price_chart,
            rsi_chart,
            equity_chart,
            metrics_box,
            llm_box
        ]
    )

# Launch
app.launch()

/tmp/ipykernel_813/3910464416.py:148: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_813/3910464416.py:148: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://361aefa79e22f4e765.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
